# DSL v3 双向追踪验证 (Bidirectional Traversal Proof)

基于 `chain_2q_native.metal.yaml` 演示四条链路：

1. **circuit → geometry**：`circuit.Q*.pad_width` 通过 `${...}` 插值进入 primitive 尺寸
2. **netlist → net_info**：`netlist.connections` 写入 `design.net_info`
3. **geometry → circuit**：从 `derived` metadata 反取出 bus path 实际长度
4. **round-trip**：`overrides` 改 circuit 后重建，geometry 同步变化


In [ ]:
from pathlib import Path
import sys

repo = Path.cwd().parents[1] if Path.cwd().name == 'dsl' else Path.cwd()
src = repo / 'src'
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

from qiskit_metal.toolbox_metal.dsl import build_design

yaml_path = Path('chain_2q_native.metal.yaml').resolve()
design = build_design(yaml_path)
chain = design.metadata['dsl_chain']
derived = chain['derived']
design

## 1. circuit → geometry

`pad_left` primitive 的尺寸由 `${circuit.Q1.pad_width}` 插值得到。

In [ ]:
def pad_left_x_extent(qubit):
    b = derived['circuit']['geometry'][qubit]['primitives']['pad_left']['bounds']
    return b[2] - b[0]

pad_circuit = chain['circuit']['Q1']['pad_width']
pad_geom = pad_left_x_extent('Q1')
assert abs(pad_geom - 0.42) < 1e-9
print(f'circuit.Q1.pad_width={pad_circuit}, geometry pad_left x-span={pad_geom} mm')

## 2. netlist → net_info


In [ ]:
assert len(design.net_info) > 0
design.net_info

## 3. geometry → circuit (extractor)

把 `derived` 里的 bus path 长度写回 `chain.circuit`。

In [ ]:
bus_length = derived['circuit']['geometry']['bus']['primitives']['center_trace']['length']
chain['circuit'].setdefault('bus', {})['actual_length'] = f'{bus_length}mm'
print(f'bus center_trace length = {bus_length} mm')

## 4. Round-trip：override circuit 后 geometry 同步变化


In [ ]:
design2 = build_design(yaml_path, overrides={'circuit': {'Q1': {'pad_width': '500um'}}})
derived2 = design2.metadata['dsl_chain']['derived']
b = derived2['circuit']['geometry']['Q1']['primitives']['pad_left']['bounds']
pad_geom2 = b[2] - b[0]
assert abs(pad_geom2 - 0.5) < 1e-9
print(f'after override Q1.pad_width=500um, geometry pad_left x-span={pad_geom2} mm')

print('PASS: circuit -> geometry')
print('PASS: netlist  -> net_info')
print('PASS: geometry -> circuit (extractor)')
print('PASS: circuit override -> geometry (round-trip)')